In [1]:
import torch
from pathlib import Path

In [2]:
# Get checkpoints files from the range 
ckpt_dir = Path("/data/ratna/from_scratch/eval") # RSG - changed from /data/OM_checkpoints , /data/ratna_retrain/eval
steps = list(range(87500, 137501, 12500))  # RSG - change this for a new range
ckpt_paths = [ ckpt_dir / f"training_{step}" / "teacher_checkpoint.pth" for step in steps]
print(f"Checkpoints to average: {steps}")
print(f"Number of paths: {len(ckpt_paths)}")

Checkpoints to average: [87500, 100000, 112500, 125000, 137500]
Number of paths: 5


In [3]:
# Filter and load checkpoints
state_dicts = [torch.load(str(p), map_location='cpu') for p in ckpt_paths if p.exists()]
print(f"Loaded {len(state_dicts)} checkpoints")

Loaded 5 checkpoints


In [4]:
teacher_dicts = [sd["teacher"] for sd in state_dicts]

In [5]:
print(len(teacher_dicts), teacher_dicts[0])

5 OrderedDict([('backbone.cls_token', tensor([[[ 0.0008, -0.0013,  0.0061,  ...,  0.0074, -0.0008,  0.0046]]])), ('backbone.pos_embed', tensor([[[ 0.0130,  0.0050, -0.0080,  ...,  0.0086,  0.0217,  0.0061],
         [ 0.0009, -0.0412, -0.0173,  ...,  0.0314,  0.0014,  0.0119],
         [ 0.0262, -0.0214, -0.0125,  ...,  0.0496,  0.0081,  0.0233],
         ...,
         [ 0.0296, -0.0073, -0.0069,  ...,  0.0265, -0.0015, -0.0567],
         [ 0.0414, -0.0105,  0.0067,  ...,  0.0247,  0.0036, -0.0239],
         [ 0.0143, -0.0313, -0.0134,  ...,  0.0426,  0.0059, -0.0042]]])), ('backbone.register_tokens', tensor([[[ 3.0319e-03, -1.3504e-02,  1.7962e-02,  ...,  1.1738e-02,
           4.6090e-03,  1.7628e-03],
         [-4.5759e-03, -8.0008e-03,  4.8602e-03,  ...,  9.9183e-04,
          -1.0694e-02, -9.3015e-04],
         [-8.4652e-03, -7.8531e-03, -1.5354e-02,  ...,  8.9012e-03,
          -9.7822e-03, -2.8716e-03],
         [-4.1395e-04, -7.4098e-04,  3.4175e-03,  ..., -5.8693e-04,
        

In [6]:
averaged_state_dict = {}
for key in teacher_dicts[0].keys():
    averaged_state_dict[key] = sum(td[key] for td in teacher_dicts) / len(teacher_dicts)

In [7]:
# Wrap back in the original structure
output_dict = {'teacher': averaged_state_dict}
print(f"Average dtype: {output_dict['teacher']['backbone.cls_token'].dtype}")

Average dtype: torch.float32


In [8]:
# Save averaged checkpoint
output_dir = Path("/data/ratna/from_scratch/eval/averaged_87500_to_137500") # RSG - change this for a new range
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "teacher_checkpoint.pth"
torch.save(output_dict, output_path)
print(f"Saved to: {output_path}")

Saved to: /data/ratna/from_scratch/eval/averaged_87500_to_137500/teacher_checkpoint.pth


The End

In [10]:
# ckpt_path = Path("/data/ratna/from_scratch/eval/averaged_87500_to_137500/teacher_checkpoint.pth")
# state_dict = torch.load(ckpt_path, map_location="cpu")
# print(state_dict)